In [1]:
"""
Pipeline Gold - Sécurité v2 (Commune + Département — dims partagées)
=====================================================================
Les deux datasets partagent les mêmes indicateurs → une seule dim_indicateur_securite.

Sorties :
  gold/security/
    ├── dim_indicateur_securite.csv   ← indicateurs fusionnés (commune + dep)
    ├── dim_departement.csv
    ├── fact_demographie.csv          ← grain : id_commune + annee
    ├── fact_securite.csv             ← grain : id_commune + annee + id_indicateur_securite
    ├── fact_demographie_dep.csv      ← grain : id_departement + annee
    └── fact_securite_dep.csv         ← grain : id_departement + annee + id_indicateur_securite
"""

from pathlib import Path
import pandas as pd

# ─────────────────────────────────────────────
# 0. CHEMINS
# ─────────────────────────────────────────────
SILVER_COMMUNE_PATH = Path("../../data/silver/securite_data_silver_clean.csv")
SILVER_DEP_PATH     = Path("../../data/silver/silver_DEP-departementale.csv")
GOLD_DIR            = Path("../../data/gold/security")
GOLD_DIR.mkdir(parents=True, exist_ok=True)


# ═══════════════════════════════════════════════════════════════════
# 1. CHARGEMENT & NORMALISATION
# ═══════════════════════════════════════════════════════════════════
print("=" * 60)
print("1. CHARGEMENT & NORMALISATION")
print("=" * 60)

# ── Commune ────────────────────────────────────────────────────────
df_sec = pd.read_csv(SILVER_COMMUNE_PATH, sep=";", dtype=str)
print(f"  Commune brut : {df_sec.shape}")

df_sec["code_departement"]   = df_sec["code_departement"].str.strip().str.zfill(2)
df_sec["code_commune"]       = df_sec["code_commune"].str.strip().str.zfill(3)
df_sec["code_insee_commune"] = df_sec["code_insee_commune"].str.strip().str.zfill(5)

df_sec["annee"]                  = pd.to_numeric(df_sec["annee"], errors="coerce").astype("Int64")
df_sec["nombre"]                 = pd.to_numeric(df_sec["nombre"], errors="coerce")
df_sec["taux_pour_mille"]        = pd.to_numeric(df_sec["taux_pour_mille"], errors="coerce")
df_sec["insee_pop"]              = pd.to_numeric(df_sec["insee_pop"], errors="coerce").astype("Int64")
df_sec["insee_log"]              = pd.to_numeric(df_sec["insee_log"], errors="coerce").astype("Int64")
df_sec["complement_info_nombre"] = pd.to_numeric(df_sec["complement_info_nombre"], errors="coerce")
df_sec["complement_info_taux"]   = pd.to_numeric(df_sec["complement_info_taux"], errors="coerce")
df_sec["source_dataset_timestamp"] = pd.to_datetime(df_sec["source_dataset_timestamp"], errors="coerce")
df_sec["date_refresh"]             = pd.to_datetime(df_sec["date_refresh"], errors="coerce")

df_sec["indicateur"]      = df_sec["indicateur"].str.strip()
df_sec["unite_de_compte"] = df_sec["unite_de_compte"].str.strip()
df_sec["est_diffuse"]     = df_sec["est_diffuse"].str.strip()
df_sec["source_dataset"]  = df_sec["source_dataset"].str.strip()

df_sec["id_commune"] = df_sec["code_departement"] + df_sec["code_commune"]

# ── Département ────────────────────────────────────────────────────
df_dep = pd.read_csv(SILVER_DEP_PATH, sep=";", dtype=str)
print(f"  Département brut : {df_dep.shape}")

df_dep["Code_departement"] = df_dep["Code_departement"].str.strip().str.zfill(2)
df_dep["Code_region"]      = df_dep["Code_region"].str.strip().str.zfill(2)

df_dep["annee"]           = pd.to_numeric(df_dep["annee"], errors="coerce").astype("Int64")
df_dep["nombre"]          = pd.to_numeric(df_dep["nombre"], errors="coerce")
df_dep["taux_pour_mille"] = pd.to_numeric(df_dep["taux_pour_mille"], errors="coerce")
df_dep["insee_pop"]       = pd.to_numeric(df_dep["insee_pop"], errors="coerce").astype("Int64")
df_dep["insee_log"]       = pd.to_numeric(df_dep["insee_log"], errors="coerce").astype("Int64")
df_dep["ingestion_timestamp"]   = pd.to_datetime(df_dep["ingestion_timestamp"], errors="coerce")

df_dep["indicateur"]            = df_dep["indicateur"].str.strip()
df_dep["unite_de_compte"]       = df_dep["unite_de_compte"].str.strip()
df_dep["extraction_source_url"] = df_dep["extraction_source_url"].str.strip()
df_dep["source_file_name"]      = df_dep["source_file_name"].str.strip()

df_dep["id_departement"] = df_dep["Code_departement"]


# ═══════════════════════════════════════════════════════════════════
# 2. DIMENSION PARTAGÉE : dim_indicateur_securite
#    Union des indicateurs des deux datasets → un seul référentiel
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("2. dim_indicateur_securite (fusionnée)")
print("=" * 60)

indicateurs_commune = df_sec[["indicateur", "unite_de_compte"]].drop_duplicates()
indicateurs_dep     = df_dep[["indicateur", "unite_de_compte"]].drop_duplicates()

dim_indicateur_securite = (
    pd.concat([indicateurs_commune, indicateurs_dep])
    .drop_duplicates()                          # ← fusionne les doublons communs
    .sort_values(["indicateur", "unite_de_compte"])
    .reset_index(drop=True)
)
dim_indicateur_securite.insert(
    0, "id_indicateur_securite",
    [f"SEC_{i:03d}" for i in range(1, len(dim_indicateur_securite) + 1)]
)

print(f"  Indicateurs commune seuls : {len(indicateurs_commune)}")
print(f"  Indicateurs dep seuls     : {len(indicateurs_dep)}")
print(f"  Après fusion (union)      : {len(dim_indicateur_securite)}")
display(dim_indicateur_securite)


# ═══════════════════════════════════════════════════════════════════
# 3. DIMENSION : dim_departement
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("3. dim_departement")
print("=" * 60)

dim_departement = (
    df_dep[["Code_departement", "Code_region"]]
    .drop_duplicates()
    .rename(columns={"Code_departement": "id_departement", "Code_region": "code_region"})
    .sort_values("id_departement")
    .reset_index(drop=True)
)
print(f"  Shape : {dim_departement.shape}")
display(dim_departement.head())


# ═══════════════════════════════════════════════════════════════════
# 4. FACTS COMMUNE
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("4. FACTS COMMUNE")
print("=" * 60)

fact_commune_full = df_sec.merge(
    dim_indicateur_securite, on=["indicateur", "unite_de_compte"], how="left"
)

# Vérification : aucun id_indicateur manquant
missing = fact_commune_full["id_indicateur_securite"].isna().sum()
if missing:
    print(f"  ⚠️  {missing} lignes sans id_indicateur_securite (commune) !")
else:
    print("  ✅ Tous les indicateurs commune ont un ID.")

# fact_demographie (grain = id_commune + annee)
fact_demographie = (
    fact_commune_full[["id_commune", "annee", "insee_pop", "insee_log"]]
    .drop_duplicates()
    .sort_values(["id_commune", "annee"])
    .reset_index(drop=True)
)
print(f"  fact_demographie  : {fact_demographie.shape}")

# fact_securite (grain = id_commune + annee + id_indicateur_securite)
fact_securite = (
    fact_commune_full[[
        "id_commune", "annee", "id_indicateur_securite",
        "nombre", "taux_pour_mille", "est_diffuse",
        "complement_info_nombre", "complement_info_taux",
        "source_dataset", "source_dataset_timestamp", "date_refresh",
    ]]
    .sort_values(["id_commune", "annee", "id_indicateur_securite"])
    .reset_index(drop=True)
)
print(f"  fact_securite     : {fact_securite.shape}")


# ═══════════════════════════════════════════════════════════════════
# 5. FACTS DÉPARTEMENT
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("5. FACTS DÉPARTEMENT")
print("=" * 60)

fact_dep_full = df_dep.merge(
    dim_indicateur_securite, on=["indicateur", "unite_de_compte"], how="left"
)

missing_dep = fact_dep_full["id_indicateur_securite"].isna().sum()
if missing_dep:
    print(f"  ⚠️  {missing_dep} lignes sans id_indicateur_securite (département) !")
else:
    print("  ✅ Tous les indicateurs département ont un ID.")

# fact_demographie_dep (grain = id_departement + annee)
fact_demographie_dep = (
    fact_dep_full[["id_departement", "annee", "insee_pop", "insee_log"]]
    .drop_duplicates()
    .sort_values(["id_departement", "annee"])
    .reset_index(drop=True)
)
print(f"  fact_demographie_dep : {fact_demographie_dep.shape}")

# fact_securite_dep (grain = id_departement + annee + id_indicateur_securite)
fact_securite_dep = (
    fact_dep_full[[
        "id_departement", "annee", "id_indicateur_securite",
        "nombre", "taux_pour_mille",
        "source_file_name", "extraction_source_url", "ingestion_timestamp",
    ]]
    .sort_values(["id_departement", "annee", "id_indicateur_securite"])
    .reset_index(drop=True)
)
print(f"  fact_securite_dep    : {fact_securite_dep.shape}")


# ═══════════════════════════════════════════════════════════════════
# 6. SAUVEGARDE GOLD
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("6. SAUVEGARDE GOLD")
print("=" * 60)

outputs = {
    "dim_indicateur_securite.csv": dim_indicateur_securite,   # ← une seule dim
    "dim_departement.csv":         dim_departement,
    "fact_demographie.csv":        fact_demographie,
    "fact_securite.csv":           fact_securite,
    "fact_demographie_dep.csv":    fact_demographie_dep,
    "fact_securite_dep.csv":       fact_securite_dep,
}

for filename, df in outputs.items():
    path = GOLD_DIR / filename
    df.to_csv(path, sep=";", index=False)
    print(f"  ✅ {filename:<40} {df.shape}")

print("\n🎉 Pipeline Gold Sécurité v2 terminé — dim_indicateur_securite unifiée.")

1. CHARGEMENT & NORMALISATION
  Commune brut : (37125, 18)
  Département brut : (180, 14)

2. dim_indicateur_securite (fusionnée)
  Indicateurs commune seuls : 15
  Indicateurs dep seuls     : 18
  Après fusion (union)      : 18


,id_indicateur_securite,indicateur,unite_de_compte
0,SEC_001,Cambriolages de logement,Infraction
1,SEC_002,Destructions et dégradations volontaires,Infraction
2,SEC_003,Escroqueries et fraudes aux moyens de paiement,Victime
3,SEC_004,Homicides,Victime
4,SEC_005,Tentatives d'homicide,Victime
5,SEC_006,Trafic de stupéfiants,Mis en cause
6,SEC_007,Usage de stupéfiants,Mis en cause
7,SEC_008,Usage de stupéfiants (AFD),Mis en cause
8,SEC_009,Usage de stupéfiants (hors AFD),Mis en cause
9,SEC_010,Violences physiques hors cadre familial,Victime



3. dim_departement
  Shape : (1, 2)


,id_departement,code_region
0,69,84



4. FACTS COMMUNE
  ✅ Tous les indicateurs commune ont un ID.
  fact_demographie  : (2475, 4)
  fact_securite     : (37125, 11)

5. FACTS DÉPARTEMENT
  ✅ Tous les indicateurs département ont un ID.
  fact_demographie_dep : (10, 4)
  fact_securite_dep    : (180, 8)

6. SAUVEGARDE GOLD
  ✅ dim_indicateur_securite.csv              (18, 3)
  ✅ dim_departement.csv                      (1, 2)
  ✅ fact_demographie.csv                     (2475, 4)
  ✅ fact_securite.csv                        (37125, 11)
  ✅ fact_demographie_dep.csv                 (10, 4)
  ✅ fact_securite_dep.csv                    (180, 8)

🎉 Pipeline Gold Sécurité v2 terminé — dim_indicateur_securite unifiée.
